# **ENGG680 - Introduction to Digital Engineering**
## *Final Project: Predicting Diabetes Progression*

## Preliminary: Certificate of Work


*We, the undersigned, certify that this is our own work, which has been done expressly for this course, either without the assistance of any other party or where appropriate we have acknowledged the work of others. Further, we have read and understood the section in the university calendar on plagiarism/cheating/other academic misconduct and we are aware of the implications thereof. We request that the total mark for this assignment be distributed as follows among group members:*

|          | First Name | Last Name | Signature (Full Name, Date) | Hours | Contribution % |
|----------|------------|-----------|-----------------------------|-------|----------------|
| Member 1: | Syed Muhammad Haider Raza | Rizvi ||  | 25 |
| Member 2: | First Name | Last Name | Signature | Hours | Contribution |
| Member 3: | First Name | Last Name | Signature | Hours | Contribution |
| Member 4: | First Name | Last Name | Signature | Hours | Contribution |


In [2]:
#import necessary libraries
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns

In [24]:
#import data as a dataframe
data = pd.read_csv('diabetic_data.csv')

#print shape
print(data.shape)

#view data
data.head()


# dataset obtained from UCI Diabetes 130 US hospitals dataset: 
# [1] J. Clore, K. Cios, J. DeShazo, and B. Strack. "Diabetes 130-US Hospitals for Years 1999-2008," 
# UCI Machine Learning Repository, 2014. [Online]. Available: https://doi.org/10.24432/C5230J.

(101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [36]:
#data preprocessing


#getting column names:
list(data.columns)

#columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 
# 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 
# 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses',
#  'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 
# 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', # 'insulin', 
# 'glyburide-metformin', # 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']

cols_to_remove = ['race','gender','weight','encounter_id', 'patient_nbr', 'payer_code', 'medical_specialty',
    'acetohexamide','tolbutamide', 'troglitazone', 'tolazamide',
    'examide', 'citoglipton', 'diag_1', 'diag_2', 'diag_3']

df_clean = data.drop(columns=cols_to_remove) #removing extra unnecessary columns
print(list(df_clean.columns))

df = df_clean.replace("?", np.nan)


df

['age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,miglitol,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,[0-10),6,25,1,1,41,0,1,0,0,...,No,No,No,No,No,No,No,No,No,NO
1,[10-20),1,1,7,3,59,0,18,0,0,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,[20-30),1,1,7,2,11,5,13,2,0,...,No,No,No,No,No,No,No,No,Yes,NO
3,[30-40),1,1,7,2,44,1,16,0,0,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,[40-50),1,1,7,1,51,0,8,0,0,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,[70-80),1,3,7,3,51,0,16,0,0,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,[80-90),1,4,5,5,33,3,18,0,0,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,[70-80),1,1,7,1,53,0,9,1,0,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,[80-90),2,3,7,10,45,2,21,0,0,...,No,Up,No,No,No,No,No,Ch,Yes,NO
